In [1]:
import pandas as pd

#load the main dataset
df = pd.read_csv("diabetic_data.csv")
print("Shape:", df.shape)
df.head()

Shape: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

In [3]:
# '?' is used as the missing value marker.
# confirm which columns need the missing-value cleanup.
print('Shape:', df.shape)
(df == '?').sum().sort_values(ascending=False).head(10)

Shape: (101766, 50)


weight               98569
medical_specialty    49949
payer_code           40256
race                  2273
diag_3                1423
diag_2                 358
diag_1                  21
admission_type_id        0
patient_nbr              0
encounter_id             0
dtype: int64

In [4]:
# dropping columns that are overwhelmingly missing or not useful for this analysis
df_clean = df.drop(columns = ['weight', 'payer_code'])

# replacing remaining '?' with 'NaN' so pandas treats them properly
df_clean = df_clean.replace('?', pd.NA)

print(df_clean.shape)
df_clean.isna().sum().sort_values(ascending = False).head(10)

(101766, 48)


max_glu_serum          96420
A1Cresult              84748
medical_specialty      49949
race                    2273
diag_3                  1423
diag_2                   358
diag_1                    21
patient_nbr                0
admission_source_id        0
time_in_hospital           0
dtype: int64

**Glycemic control [max_glu_serum(maximum serum glucose) and HBA1C(glycated hemoglobin) are not 
always tested during a routine hospital stay unless for a specific reason. So it is genuinely not tested for in most cases.
This reason accounts for the sum of null in those two columns being high.**

In [5]:
# replacing 'Null' in race colunm with 'Unkonwn'
df_clean['race'] = df_clean['race'].fillna('Unknown')

# considering medical_specialty: keep top categories, 'Missing' for NaN, and
# 'Other' for non-frequently used categories.
top_specialty = df_clean['medical_specialty'].value_counts().head(10).index

    # using lambda function to effect changes in medical_specialty
df_clean['medical_specialty'] = df_clean['medical_specialty'].apply(lambda x: x if x in top_specialty
                                                                   else ('Missing' if pd.isna(x) else
                                                                        'Other'))
df_clean['medical_specialty'].value_counts()

medical_specialty
Missing                       49949
InternalMedicine              14635
Emergency/Trauma               7565
Other                          7469
Family/GeneralPractice         7440
Cardiology                     5352
Surgery-General                3099
Nephrology                     1613
Orthopedics                    1400
Orthopedics-Reconstructive     1233
Radiologist                    1140
Pulmonology                     871
Name: count, dtype: int64

In [6]:
# creating a function that maps diag_1, diag_2, and diag_3 to their respective clinical category
def map_diag(code):
    if pd.isna(code):
        return 'Missing' # Mark clinical category as Missing when code is Null, None or NaN
        
    #convert diag_1, diag_2 and diag_3 values to string since they contain alphanumeric values
    code =str(code)

    # representing supplementary/external codes(V and E) as 'Other'
    if code.startswith('V') or code.startswith('E'):
        return 'Other'
    # trying to convert values to float and returning 'Other' for malformed data
    try:
        code = float(code)
    except ValueError:
        return 'Other'

    if 250 <= code < 251:
        return 'Diabetes'
    elif 390 <= code <= 459 or code == 785:
        return 'Circulatory'
    elif 460 <= code <= 519 or code == 786:
        return 'Respiratory'
    elif 520 <= code <= 579 or code == 787:
        return 'Digestive'
    elif 800 <= code <= 999:
        return 'Injury'
    elif 710 <= code <= 739:
        return 'Musculoskeletal'
    elif 580 <= code <= 629 or code == 788:
        return 'Genitourinary'
    elif 140 <= code <= 239:
        return 'Neoplasms'
    else:
        return 'Other'

# updating colunm name for the three colunms and applying the map_diag function
for col in ['diag_1', 'diag_2', 'diag_3']:
    df_clean[col + '_group'] = df_clean[col].apply(map_diag)

df_clean[['diag_1_group', 'diag_2_group', 'diag_3_group']].head(10)

,diag_1_group,diag_2_group,diag_3_group
0,Diabetes,Missing,Missing
1,Other,Diabetes,Other
2,Other,Diabetes,Other
3,Other,Diabetes,Circulatory
4,Neoplasms,Neoplasms,Diabetes
5,Circulatory,Circulatory,Diabetes
6,Circulatory,Circulatory,Other
7,Circulatory,Respiratory,Diabetes
8,Circulatory,Circulatory,Other
9,Circulatory,Neoplasms,Respiratory


In [7]:
df_clean['age'].head(10)

# converting age bins to midpoint(average)
def average_age(age_bin):
    first, second = age_bin.strip('[)').split('-')
    return (int(first) + int(second))/2

# effect changes but maintain age bins
df_clean['age_midpoint'] = df_clean['age'].apply(average_age)
df_clean[['age','age_midpoint']].head(10)

,age,age_midpoint
0,[0-10),5.0
1,[10-20),15.0
2,[20-30),25.0
3,[30-40),35.0
4,[40-50),45.0
5,[50-60),55.0
6,[60-70),65.0
7,[70-80),75.0
8,[80-90),85.0
9,[90-100),95.0


In [8]:
# loading IDS_mapping dataset to filter out expired, hospice discharges
ids_map = pd.read_csv('IDS_mapping.csv')

# checking values
df_clean['discharge_disposition_id'].value_counts().head(20)

discharge_disposition_id
1     60234
3     13954
6     12902
18     3691
2      2128
22     1993
11     1642
5      1184
25      989
4       815
7       623
23      412
13      399
14      372
28      139
8       108
15       63
24       48
9        21
17       14
Name: count, dtype: int64

In [9]:
# confirming the meaning of the 'discharge_disposition_id' with 'expired' or 'hospice' in their description
disposition_map = ids_map.iloc[:35].copy()
disposition_map.columns = ['discharge_disposition_id', 'description']
disposition_map[disposition_map['description'].str.contains('Expired|Hospice', case=False, na=False)]

,discharge_disposition_id,description
20,11,Expired
22,13,Hospice / home
23,14,Hospice / medical facility
28,19,"Expired at home. Medicaid only, hospice."
29,20,"Expired in a medical facility. Medicaid only, ..."
30,21,"Expired, place unknown. Medicaid only, hospice."


In [10]:
# removing rows where discharge_disposition_id's description has 'expired' or 'Hospice'
excluded_ids = [11,13,14,19,20,21]
df_clean = df_clean[~df_clean['discharge_disposition_id'].isin(excluded_ids)]
print(df_clean.shape)

(99343, 52)


In [16]:
# grouping readmission based on the number of days after discharge. 
#True=1=readmission within 30 days
#False=0=No readmission within 30 days.

df_clean['readmitted_30'] = (df_clean['readmitted'] == '<30').astype(int)
df_clean['readmitted_30'].value_counts()

readmitted_30
0    88029
1    11314
Name: count, dtype: int64

In [17]:
# sorting by encounter_id
df_clean = df_clean.sort_values('encounter_id')

# now drop duplicates based on patient_nbr while maintaining the first occurence
df_clean = df_clean.drop_duplicates(subset= 'patient_nbr', keep = 'first')
print(df_clean.shape)

(69990, 53)


In [18]:
# saving cleaned dataset
df_clean.to_csv('cleaned_diabetic_data.csv', index=False)